In [72]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [73]:
!ls -la /content/drive/MyDrive/Ciencia\ de\ Datos/Riesgos

total 2827
-rw------- 1 root root    1195 Mar 12 20:39 label_encoders.joblib
-rw------- 1 root root    1279 Mar 12 20:38 minmax_scaler.joblib
-rw------- 1 root root    2775 Mar 12 20:38 pca_model_8_componentes.joblib
-rw------- 1 root root 2887904 Mar 12 17:46 riesgo.xlsx


In [74]:
%cd /content/drive/MyDrive/Ciencia\ de\ Datos/Riesgos

/content/drive/MyDrive/Ciencia de Datos/Riesgos


In [75]:
import pandas as pd

df = pd.read_excel("riesgo.xlsx")
df.head()

,Customer_ID,Name,Age,SSN,Occupation,Annual_Income,Monthly_Inhand_Salary,Num_Bank_Accounts,Num_Credit_Card,Interest_Rate,...,Credit_Mix,Outstanding_Debt,Credit_Utilization_Ratio,Credit_History_Age,Payment_of_Min_Amount,Total_EMI_per_month,Amount_invested_monthly,Payment_Behaviour,Monthly_Balance,Credit_Score
0,CUS_0x1000,Alistair Barrf,17.375,913-74-1218,Lawyer,30625.94,2706.161667,6.0,5.0,27,...,Bad,1562.91,33.477546,10.458333,Yes,42.941090,158.549735,High_spent_Medium_value_payments,335.375341,0
1,CUS_0x1009,Arunah,25.750,063-67-6938,Mechanic,52312.68,4250.390000,6.0,5.0,17,...,Standard,202.68,29.839984,30.714286,Yes,108.366467,146.679378,High_spent_Medium_value_payments,428.743155,1
2,CUS_0x100b,Shirboni,18.500,238-62-0395,Media_Manager,113781.39,9549.782500,1.0,4.0,1,...,Good,1030.20,34.841449,15.571429,No,0.000000,505.386526,High_spent_Large_value_payments,781.229776,0
3,CUS_0x1011,Schneyerh,43.875,793-05-8223,Doctor,58918.47,5208.872500,3.0,3.0,17,...,Standard,473.14,27.655897,15.541667,Yes,123.434939,311.060914,Low_spent_Medium_value_payments,332.642837,1
4,CUS_0x1013,Cameront,43.750,930-49-9615,Mechanic,98620.98,7962.415000,3.0,3.0,6,...,Good,1233.51,31.933940,17.535714,No,228.018084,355.442408,High_spent_Medium_value_payments,472.781009,1


In [76]:
# Display null counts and data types for each column
# Nota: También se puede hacer más sencillo con df.info(), pero aquí se muestra un DataFrame personalizado para mayor claridad.
null_info = pd.DataFrame({
    'Column': df.columns,
    'Data_Type': df.dtypes.values,
    'Null_Count': df.isnull().sum().values,
    'Null_Percentage': (df.isnull().sum().values / len(df) * 100).round(2)
})

print(null_info)

                      Column Data_Type  Null_Count  Null_Percentage
0                Customer_ID    object           0             0.00
1                       Name    object           0             0.00
2                        Age   float64           0             0.00
3                        SSN    object           0             0.00
4                 Occupation    object           0             0.00
5              Annual_Income   float64           0             0.00
6      Monthly_Inhand_Salary   float64           0             0.00
7          Num_Bank_Accounts   float64           0             0.00
8            Num_Credit_Card   float64           0             0.00
9              Interest_Rate     int64           0             0.00
10               Num_of_Loan     int64           0             0.00
11              Type_of_Loan    object        1426            11.41
12       Delay_from_due_date   float64           0             0.00
13    Num_of_Delayed_Payment   float64          

In [77]:
# Eliminar las columnas con identificadores personales y la columna de tipo de préstamo
# ya que tiene muchos nulos e información difícil de procesar.
df = df.drop(columns=['Customer_ID', 'Name', 'SSN', 'Type_of_Loan'])

df.head()

,Age,Occupation,Annual_Income,Monthly_Inhand_Salary,Num_Bank_Accounts,Num_Credit_Card,Interest_Rate,Num_of_Loan,Delay_from_due_date,Num_of_Delayed_Payment,...,Credit_Mix,Outstanding_Debt,Credit_Utilization_Ratio,Credit_History_Age,Payment_of_Min_Amount,Total_EMI_per_month,Amount_invested_monthly,Payment_Behaviour,Monthly_Balance,Credit_Score
0,17.375,Lawyer,30625.94,2706.161667,6.0,5.0,27,2,62.25,25.000000,...,Bad,1562.91,33.477546,10.458333,Yes,42.941090,158.549735,High_spent_Medium_value_payments,335.375341,0
1,25.750,Mechanic,52312.68,4250.390000,6.0,5.0,17,4,7.25,17.857143,...,Standard,202.68,29.839984,30.714286,Yes,108.366467,146.679378,High_spent_Medium_value_payments,428.743155,1
2,18.500,Media_Manager,113781.39,9549.782500,1.0,4.0,1,0,13.50,7.375000,...,Good,1030.20,34.841449,15.571429,No,0.000000,505.386526,High_spent_Large_value_payments,781.229776,0
3,43.875,Doctor,58918.47,5208.872500,3.0,3.0,17,3,27.25,14.500000,...,Standard,473.14,27.655897,15.541667,Yes,123.434939,311.060914,Low_spent_Medium_value_payments,332.642837,1
4,43.750,Mechanic,98620.98,7962.415000,3.0,3.0,6,3,12.50,8.428571,...,Good,1233.51,31.933940,17.535714,No,228.018084,355.442408,High_spent_Medium_value_payments,472.781009,1


In [78]:
df.describe(include='object').T

,count,unique,top,freq
Occupation,12500,15,Lawyer,887
Credit_Mix,12500,3,Standard,5731
Payment_of_Min_Amount,12500,3,Yes,7360
Payment_Behaviour,12500,6,Low_spent_Small_value_payments,3860


In [79]:
# Eliminar la columna Ocupación ya que tiene una cardinalidad muy alta
df = df.drop(columns=['Occupation'])

In [80]:
df = df.reset_index(drop=True)

# Verificar
df.head()

,Age,Annual_Income,Monthly_Inhand_Salary,Num_Bank_Accounts,Num_Credit_Card,Interest_Rate,Num_of_Loan,Delay_from_due_date,Num_of_Delayed_Payment,Changed_Credit_Limit,...,Credit_Mix,Outstanding_Debt,Credit_Utilization_Ratio,Credit_History_Age,Payment_of_Min_Amount,Total_EMI_per_month,Amount_invested_monthly,Payment_Behaviour,Monthly_Balance,Credit_Score
0,17.375,30625.94,2706.161667,6.0,5.0,27,2,62.25,25.000000,1.880000,...,Bad,1562.91,33.477546,10.458333,Yes,42.941090,158.549735,High_spent_Medium_value_payments,335.375341,0
1,25.750,52312.68,4250.390000,6.0,5.0,17,4,7.25,17.857143,9.730000,...,Standard,202.68,29.839984,30.714286,Yes,108.366467,146.679378,High_spent_Medium_value_payments,428.743155,1
2,18.500,113781.39,9549.782500,1.0,4.0,1,0,13.50,7.375000,10.911429,...,Good,1030.20,34.841449,15.571429,No,0.000000,505.386526,High_spent_Large_value_payments,781.229776,0
3,43.875,58918.47,5208.872500,3.0,3.0,17,3,27.25,14.500000,14.170000,...,Standard,473.14,27.655897,15.541667,Yes,123.434939,311.060914,Low_spent_Medium_value_payments,332.642837,1
4,43.750,98620.98,7962.415000,3.0,3.0,6,3,12.50,8.428571,1.705000,...,Good,1233.51,31.933940,17.535714,No,228.018084,355.442408,High_spent_Medium_value_payments,472.781009,1


In [81]:
# Separar columnas por tipo
object_cols = df.select_dtypes(include="object").columns.tolist()
numeric_cols = df.select_dtypes(include="number").columns.tolist()

print("Columnas object:", object_cols)
print("Columnas numéricas:", numeric_cols)

Columnas object: ['Credit_Mix', 'Payment_of_Min_Amount', 'Payment_Behaviour']
Columnas numéricas: ['Age', 'Annual_Income', 'Monthly_Inhand_Salary', 'Num_Bank_Accounts', 'Num_Credit_Card', 'Interest_Rate', 'Num_of_Loan', 'Delay_from_due_date', 'Num_of_Delayed_Payment', 'Changed_Credit_Limit', 'Num_Credit_Inquiries', 'Outstanding_Debt', 'Credit_Utilization_Ratio', 'Credit_History_Age', 'Total_EMI_per_month', 'Amount_invested_monthly', 'Monthly_Balance', 'Credit_Score']


In [82]:
for col in object_cols:
    unique_values = df[col].unique()
    print(f"Columna: {col}")
    print(f"Valores únicos: {unique_values}")
    print("-" * 30)

Columna: Credit_Mix
Valores únicos: ['Bad' 'Standard' 'Good']
------------------------------
Columna: Payment_of_Min_Amount
Valores únicos: ['Yes' 'No' 'NM']
------------------------------
Columna: Payment_Behaviour
Valores únicos: ['High_spent_Medium_value_payments' 'High_spent_Large_value_payments'
 'Low_spent_Medium_value_payments' 'Low_spent_Small_value_payments'
 'High_spent_Small_value_payments' 'Low_spent_Large_value_payments']
------------------------------


In [83]:
from sklearn.preprocessing import LabelEncoder

df_limpio = df.copy()
le = LabelEncoder()

for c in object_cols:
    df_limpio[c] = le.fit_transform(df_limpio[c].astype(str))

df_limpio.head()

,Age,Annual_Income,Monthly_Inhand_Salary,Num_Bank_Accounts,Num_Credit_Card,Interest_Rate,Num_of_Loan,Delay_from_due_date,Num_of_Delayed_Payment,Changed_Credit_Limit,...,Credit_Mix,Outstanding_Debt,Credit_Utilization_Ratio,Credit_History_Age,Payment_of_Min_Amount,Total_EMI_per_month,Amount_invested_monthly,Payment_Behaviour,Monthly_Balance,Credit_Score
0,17.375,30625.94,2706.161667,6.0,5.0,27,2,62.25,25.000000,1.880000,...,0,1562.91,33.477546,10.458333,2,42.941090,158.549735,1,335.375341,0
1,25.750,52312.68,4250.390000,6.0,5.0,17,4,7.25,17.857143,9.730000,...,2,202.68,29.839984,30.714286,2,108.366467,146.679378,1,428.743155,1
2,18.500,113781.39,9549.782500,1.0,4.0,1,0,13.50,7.375000,10.911429,...,1,1030.20,34.841449,15.571429,1,0.000000,505.386526,0,781.229776,0
3,43.875,58918.47,5208.872500,3.0,3.0,17,3,27.25,14.500000,14.170000,...,2,473.14,27.655897,15.541667,2,123.434939,311.060914,4,332.642837,1
4,43.750,98620.98,7962.415000,3.0,3.0,6,3,12.50,8.428571,1.705000,...,1,1233.51,31.933940,17.535714,1,228.018084,355.442408,1,472.781009,1


In [84]:
df_limpio.shape

(12500, 21)

In [85]:
import joblib

# La celda anterior ya aplicó fit_transform a df_limpio. 
# Para poder guardar los encoders, necesitamos haberlos guardado en un diccionario durante ese proceso.
# Si no se guardaron, debemos recrear el diccionario de encoders ajustados:

label_encoders = {}
for c in object_cols:
    from sklearn.preprocessing import LabelEncoder
    le_col = LabelEncoder()
    le_col.fit(df[c].astype(str))
    label_encoders[c] = le_col

# Guardar los encoders ajustados
joblib.dump(label_encoders, "label_encoders.joblib")
print("LabelEncoders guardados exitosamente.")

LabelEncoders guardados exitosamente.


In [109]:
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import train_test_split

# Definir X e y (robusto a notebook corrido con columnas en inglés o español)
target_col_model = "Credit_Score" if "Credit_Score" in df_limpio.columns else "Puntaje_Credito"
X = df_limpio.drop(columns=[target_col_model])
y = df_limpio[target_col_model]
y_original = y.copy()

# Split en bruto ANTES de cualquier fit de preprocesamiento
X_train_raw, X_test_raw, y_train_raw, y_test_raw = train_test_split(
    X,
    y,
    test_size=0.15,
    random_state=32,
    stratify=y_original,
)

# Selección de features ajustada solo en train
selector = SelectKBest(score_func=f_classif, k=15)
selector.fit(X_train_raw, y_train_raw)
selected_features = X.columns[selector.get_support()].tolist()

# Transformar train y test con el selector entrenado
X_train_selected = pd.DataFrame(
    selector.transform(X_train_raw),
    columns=selected_features,
    index=X_train_raw.index,
)
X_test_selected = pd.DataFrame(
    selector.transform(X_test_raw),
    columns=selected_features,
    index=X_test_raw.index,
)

print(f"Tamaño entrenamiento: {X_train_raw.shape[0]} muestras ({X_train_raw.shape[0]/len(X)*100:.1f}%)")
print(f"Tamaño prueba:        {X_test_raw.shape[0]} muestras ({X_test_raw.shape[0]/len(X)*100:.1f}%)")
print("Features seleccionadas:")
selected_features

Tamaño entrenamiento: 10625 muestras (85.0%)
Tamaño prueba:        1875 muestras (15.0%)
Features seleccionadas:


['Ingreso_Anual',
 'Salario_Mensual_Neto',
 'Num_Cuentas_Bancarias',
 'Num_Tarjetas_Credito',
 'Tasa_Interes',
 'Num_Prestamos',
 'Retraso_Fecha_Vencimiento',
 'Num_Pagos_Atrasados',
 'Cambio_Limite_Credito',
 'Num_Consultas_Credito',
 'Mezcla_Credito',
 'Deuda_Pendiente',
 'Antiguedad_Historial_Credito',
 'Pago_Minimo',
 'Balance_Mensual']

In [87]:
# Renombrar columnas a español
columnas_es = {
    "Age": "Edad",
    "Annual_Income": "Ingreso_Anual",
    "Monthly_Inhand_Salary": "Salario_Mensual_Neto",
    "Num_Bank_Accounts": "Num_Cuentas_Bancarias",
    "Num_Credit_Card": "Num_Tarjetas_Credito",
    "Interest_Rate": "Tasa_Interes",
    "Num_of_Loan": "Num_Prestamos",
    "Delay_from_due_date": "Retraso_Fecha_Vencimiento",
    "Num_of_Delayed_Payment": "Num_Pagos_Atrasados",
    "Changed_Credit_Limit": "Cambio_Limite_Credito",
    "Num_Credit_Inquiries": "Num_Consultas_Credito",
    "Credit_Mix": "Mezcla_Credito",
    "Outstanding_Debt": "Deuda_Pendiente",
    "Credit_Utilization_Ratio": "Ratio_Utilizacion_Credito",
    "Credit_History_Age": "Antiguedad_Historial_Credito",
    "Payment_of_Min_Amount": "Pago_Minimo",
    "Total_EMI_per_month": "EMI_Total_Mensual",
    "Amount_invested_monthly": "Monto_Invertido_Mensual",
    "Payment_Behaviour": "Comportamiento_Pago",
    "Monthly_Balance": "Balance_Mensual",
    "Credit_Score": "Puntaje_Credito",
}

# Aplicar la traducción al dataframe que estamos usando para el modelo
df_limpio = df_limpio.rename(columns=columnas_es)

# Actualizar la lista de features seleccionadas con los nuevos nombres
selected_features_es = [columnas_es.get(col, col) for col in selected_features]

df_limpio.head()

,Edad,Ingreso_Anual,Salario_Mensual_Neto,Num_Cuentas_Bancarias,Num_Tarjetas_Credito,Tasa_Interes,Num_Prestamos,Retraso_Fecha_Vencimiento,Num_Pagos_Atrasados,Cambio_Limite_Credito,...,Mezcla_Credito,Deuda_Pendiente,Ratio_Utilizacion_Credito,Antiguedad_Historial_Credito,Pago_Minimo,EMI_Total_Mensual,Monto_Invertido_Mensual,Comportamiento_Pago,Balance_Mensual,Puntaje_Credito
0,17.375,30625.94,2706.161667,6.0,5.0,27,2,62.25,25.000000,1.880000,...,0,1562.91,33.477546,10.458333,2,42.941090,158.549735,1,335.375341,0
1,25.750,52312.68,4250.390000,6.0,5.0,17,4,7.25,17.857143,9.730000,...,2,202.68,29.839984,30.714286,2,108.366467,146.679378,1,428.743155,1
2,18.500,113781.39,9549.782500,1.0,4.0,1,0,13.50,7.375000,10.911429,...,1,1030.20,34.841449,15.571429,1,0.000000,505.386526,0,781.229776,0
3,43.875,58918.47,5208.872500,3.0,3.0,17,3,27.25,14.500000,14.170000,...,2,473.14,27.655897,15.541667,2,123.434939,311.060914,4,332.642837,1
4,43.750,98620.98,7962.415000,3.0,3.0,6,3,12.50,8.428571,1.705000,...,1,1233.51,31.933940,17.535714,1,228.018084,355.442408,1,472.781009,1


In [111]:
from sklearn.preprocessing import MinMaxScaler

# Renombrar columnas seleccionadas a español para consistencia visual
X_train_selected_es = X_train_selected.rename(columns=columnas_es)
X_test_selected_es = X_test_selected.rename(columns=columnas_es)

# Crear y ajustar scaler SOLO con train
scaler = MinMaxScaler()
X_train_normalized = scaler.fit_transform(X_train_selected_es)
X_test_normalized = scaler.transform(X_test_selected_es)

# Guardar el scaler
joblib.dump(scaler, "minmax_scaler.joblib")
print("MinMaxScaler guardado exitosamente.")

# Convertir a DataFrame para mejor visualización
selected_features_es = X_train_selected_es.columns.tolist()
X_train_normalized_df = pd.DataFrame(X_train_normalized, columns=selected_features_es, index=X_train_selected_es.index)
X_test_normalized_df = pd.DataFrame(X_test_normalized, columns=selected_features_es, index=X_test_selected_es.index)
X_train_normalized_df.head()

MinMaxScaler guardado exitosamente.


,Ingreso_Anual,Salario_Mensual_Neto,Num_Cuentas_Bancarias,Num_Tarjetas_Credito,Tasa_Interes,Num_Prestamos,Retraso_Fecha_Vencimiento,Num_Pagos_Atrasados,Cambio_Limite_Credito,Num_Consultas_Credito,Mezcla_Credito,Deuda_Pendiente,Antiguedad_Historial_Credito,Pago_Minimo,Balance_Mensual
3646,0.449988,0.464086,0.380952,0.240964,0.272727,0.333333,0.126437,0.393365,0.261963,0.213740,0.5,0.001010,0.731421,0.5,0.343009
2286,0.006052,0.044321,0.761905,0.626506,0.696970,0.888889,0.764368,0.882871,0.629757,0.732824,0.0,0.787866,0.186869,1.0,0.156706
344,0.281326,0.275839,0.285714,0.915663,0.757576,0.333333,0.245211,0.682464,0.271926,0.648855,1.0,0.415103,0.191739,1.0,0.237062
1373,0.240505,0.269472,0.476190,0.433735,0.121212,0.222222,0.235632,0.644550,0.481300,0.694656,1.0,0.069126,0.477092,1.0,0.238852
1355,0.077532,0.090516,0.857143,0.433735,1.000000,0.888889,0.750958,0.867299,0.788992,0.488550,0.0,0.570635,0.358586,1.0,0.134188


In [112]:
from sklearn.decomposition import PCA

# Ajustar PCA SOLO con train y transformar train/test
pca = PCA(n_components=8)
X_train_pca = pca.fit_transform(X_train_normalized_df)
X_test_pca = pca.transform(X_test_normalized_df)

# DataFrames finales para modelado
X_train = pd.DataFrame(
    X_train_pca,
    columns=[f"PC{i}" for i in range(1, 9)],
    index=X_train_normalized_df.index,
)
X_test = pd.DataFrame(
    X_test_pca,
    columns=[f"PC{i}" for i in range(1, 9)],
    index=X_test_normalized_df.index,
)

# Guardar el modelo PCA
joblib.dump(pca, "pca_model_8_componentes.joblib")
print("Modelo PCA guardado exitosamente.")

# Varianza explicada
print("Varianza explicada por componente:")
print(pca.explained_variance_ratio_)

print("Varianza explicada acumulada:")
print(pca.explained_variance_ratio_.cumsum())

X_train.head()

Modelo PCA guardado exitosamente.
Varianza explicada por componente:
[0.46591844 0.15759349 0.09556673 0.05030925 0.03452106 0.03196666
 0.02827255 0.02660951]
Varianza explicada acumulada:
[0.46591844 0.62351193 0.71907865 0.7693879  0.80390896 0.83587562
 0.86414817 0.89075768]


,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8
3646,0.527535,-0.332024,0.076441,-0.039324,0.084048,-0.103919,-0.039359,-0.140778
2286,-1.326678,-0.176274,-0.077911,-0.096028,-0.057434,-0.131730,0.019940,-0.042196
344,-0.259733,0.449579,0.231926,-0.097314,-0.007768,0.541149,-0.026528,0.145476
1373,0.169800,0.459512,0.101592,-0.109456,-0.130935,-0.068995,-0.231624,0.075425
1355,-1.235644,-0.133566,-0.006836,0.001912,-0.086770,-0.220356,0.151545,-0.350361


In [90]:
# Crear el DataFrame de pesos (loadings) con los nombres de las features en español
loadings = pd.DataFrame(
    pca.components_.T, 
    columns=[f"PC{i}" for i in range(1, 9)], 
    index=selected_features_es
)

print("Pesos (Loadings) por componente (en español):")
loadings.round(3)

Pesos (Loadings) por componente (en español):


Pesos (Loadings) por componente (en español):


,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8
Ingreso_Anual,0.150,-0.201,0.625,-0.005,0.066,-0.011,0.014,-0.002
Salario_Mensual_Neto,0.144,-0.194,0.604,-0.004,0.065,-0.011,0.014,-0.001
Num_Cuentas_Bancarias,-0.269,0.157,0.109,0.447,0.092,-0.420,-0.129,0.204
Num_Tarjetas_Credito,-0.190,0.002,0.047,0.191,-0.081,0.259,0.459,0.699
Tasa_Interes,-0.326,0.118,0.118,0.178,-0.012,0.524,0.230,-0.500
Num_Prestamos,-0.328,-0.027,0.053,-0.418,0.719,-0.217,0.328,-0.028
Retraso_Fecha_Vencimiento,-0.260,-0.002,0.072,0.316,-0.022,0.024,0.106,0.084
Num_Pagos_Atrasados,-0.256,0.133,0.087,0.349,0.060,-0.358,-0.191,-0.194
Cambio_Limite_Credito,-0.164,0.154,0.135,-0.400,-0.604,-0.401,0.307,-0.049
Num_Consultas_Credito,-0.278,0.078,0.098,-0.071,-0.020,0.316,-0.139,-0.014


In [110]:
from tensorflow.keras.utils import to_categorical

# One-hot para referencia global (se mantiene por compatibilidad con celdas de diagnóstico)
y_one_hot = to_categorical(y_original.astype(int))

# Targets definitivos para entrenamiento/evaluación
y_train = to_categorical(y_train_raw.astype(int))
y_test = to_categorical(y_test_raw.astype(int))

print("y_train original (primeros 10 valores):")
print(y_train_raw.head(10))

print("\ny_train en one-hot (primeras 10 filas):")
print(y_train[:10])

y_train original (primeros 10 valores):
3646     2
2286     1
344      1
1373     1
1355     0
2939     2
6404     0
12162    2
3709     1
2161     1
Name: Puntaje_Credito, dtype: int64

y_train en one-hot (primeras 10 filas):
[[0. 0. 1.]
 [0. 1. 0.]
 [0. 1. 0.]
 [0. 1. 0.]
 [1. 0. 0.]
 [0. 0. 1.]
 [1. 0. 0.]
 [0. 0. 1.]
 [0. 1. 0.]
 [0. 1. 0.]]


In [113]:
# El split ya se hizo al inicio del pipeline (antes de SelectKBest/Scaler/PCA)
# Aquí solo verificamos tamaños finales listos para entrenar
print(f"Tamaño entrenamiento final: {X_train.shape[0]} muestras")
print(f"Tamaño prueba final:        {X_test.shape[0]} muestras")
print(f"Dimensión de entrada (PCA): {X_train.shape[1]} features")

Tamaño entrenamiento final: 10625 muestras
Tamaño prueba final:        1875 muestras
Dimensión de entrada (PCA): 8 features


In [114]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense

# Red neuronal: input + 3 capas ocultas (64, 32, 16) con ReLU + salida softmax (3 clases)
model = Sequential([
    Input(shape=(X_train.shape[1],)),
    Dense(64, activation="relu"),
    Dense(32, activation="relu"),
    Dense(16, activation="relu"),
    Dense(3, activation="softmax")
])

# Compilación con Adam (learning rate por defecto)
model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_12 (Dense)                │ (None, 64)             │           576 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_14 (Dense)                │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_15 (Dense)                │ (None, 3)              │            51 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,235 (12.64 KB)

 Trainable params: 3,235 (12.64 KB)

 Non-trainable params: 0 (0.00 B)

In [99]:
# Entrenar la red neuronal con 50 épocas y batch de 32 usando los datos divididos
history = model.fit(
    X_train, 
    y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.15,
    verbose=1
)


Epoch 1/50
283/283 ━━━━━━━━━━━━━━━━━━━━ 5s 11ms/step - accuracy: 0.6729 - loss: 0.7495 - val_accuracy: 0.7171 - val_loss: 0.6485
Epoch 2/50
283/283 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.7308 - loss: 0.6236 - val_accuracy: 0.7227 - val_loss: 0.6444
Epoch 3/50
283/283 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7348 - loss: 0.6175 - val_accuracy: 0.7258 - val_loss: 0.6312
Epoch 4/50
283/283 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7371 - loss: 0.6153 - val_accuracy: 0.7290 - val_loss: 0.6246
Epoch 5/50
283/283 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7371 - loss: 0.6112 - val_accuracy: 0.7183 - val_loss: 0.6227
Epoch 6/50
283/283 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7377 - loss: 0.6096 - val_accuracy: 0.7208 - val_loss: 0.6268
Epoch 7/50
283/283 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7383 - loss: 0.6081 - val_accuracy: 0.7258 - val_loss: 0.6233
Epoch 8/50
283/283 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7401 - loss: 0.6068 - val_accuracy: 0

In [103]:
# Evaluar el modelo en el conjunto de prueba
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"\nResultados en el conjunto de prueba:")
print(f"Pérdida:   {test_loss:.4f}")
print(f"Precisión: {test_accuracy:.4f}")



Resultados en el conjunto de prueba:
Pérdida:   0.6004
Precisión: 0.7520


In [104]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

# Diagnóstico rápido de desempeño por clase
true_classes = np.argmax(y_test, axis=1)
pred_classes = np.argmax(model.predict(X_test, verbose=0), axis=1)

print("Distribución real en test:")
unique, counts = np.unique(true_classes, return_counts=True)
for cls, n in zip(unique, counts):
    print(f"Clase {cls}: {n} ({n/len(true_classes):.2%})")

majority_baseline = counts.max() / len(true_classes)
print(f"\nBaseline (siempre clase mayoritaria): {majority_baseline:.4f}")

print("\nMatriz de confusión:")
print(confusion_matrix(true_classes, pred_classes))

print("\nReporte de clasificación:")
print(classification_report(true_classes, pred_classes, digits=4))

Distribución real en test:
Clase 0: 624 (33.28%)
Clase 1: 917 (48.91%)
Clase 2: 334 (17.81%)

Baseline (siempre clase mayoritaria): 0.4891

Matriz de confusión:
[[404 145  75]
 [ 68 729 120]
 [  7  50 277]]

Reporte de clasificación:
              precision    recall  f1-score   support

           0     0.8434    0.6474    0.7325       624
           1     0.7890    0.7950    0.7920       917
           2     0.5869    0.8293    0.6873       334

    accuracy                         0.7520      1875
   macro avg     0.7397    0.7573    0.7373      1875
weighted avg     0.7711    0.7520    0.7536      1875



In [105]:
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense

# Prueba controlada: entrenar sin PCA (usando las 15 features seleccionadas y normalizadas)
X_train_npca, X_test_npca, y_train_npca, y_test_npca = train_test_split(
    X_normalized_df,
    y_one_hot,
    test_size=0.15,
    random_state=32,
    stratify=y_original,
)

model_npca = Sequential([
    Input(shape=(X_normalized_df.shape[1],)),
    Dense(128, activation="relu"),
    Dense(64, activation="relu"),
    Dense(32, activation="relu"),
    Dense(3, activation="softmax"),
])

model_npca.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

history_npca = model_npca.fit(
    X_train_npca,
    y_train_npca,
    epochs=30,
    batch_size=32,
    validation_split=0.15,
    verbose=0,
)

loss_npca, acc_npca = model_npca.evaluate(X_test_npca, y_test_npca, verbose=0)
print(f"Accuracy sin PCA: {acc_npca:.4f}")
print(f"Val accuracy final sin PCA: {history_npca.history['val_accuracy'][-1]:.4f}")

Accuracy sin PCA: 0.7509
Val accuracy final sin PCA: 0.7321


In [106]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Baseline clásico para contrastar con la red neuronal
rf = RandomForestClassifier(
    n_estimators=400,
    random_state=32,
    n_jobs=-1,
    class_weight="balanced_subsample",
)

rf.fit(X_train_npca, np.argmax(y_train_npca, axis=1))
rf_pred = rf.predict(X_test_npca)
rf_acc = accuracy_score(np.argmax(y_test_npca, axis=1), rf_pred)

print(f"Accuracy RandomForest: {rf_acc:.4f}")

Accuracy RandomForest: 0.7541


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pérdida
axes[0].plot(history.history['loss'], label='Entrenamiento')
axes[0].plot(history.history['val_loss'], label='Validación')
axes[0].set_title('Pérdida del Modelo')
axes[0].set_xlabel('Época')
axes[0].set_ylabel('Pérdida')
axes[0].legend()

# Precisión
axes[1].plot(history.history['accuracy'], label='Entrenamiento')
axes[1].plot(history.history['val_accuracy'], label='Validación')
axes[1].set_title('Precisión del Modelo')
axes[1].set_xlabel('Época')
axes[1].set_ylabel('Precisión')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

# Obtener predicciones
pred_classes = np.argmax(model.predict(X_test, verbose=0), axis=1)
true_classes = np.argmax(y_test, axis=1)

# Matriz de confusión
cm = confusion_matrix(true_classes, pred_classes)

# Etiquetas de clases
class_names = ["Bad (0)", "Standard (1)", "Good (2)"]

# Figura con dos subplots: heatmap + estadísticas
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Heatmap ---
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names,
    ax=axes[0],
    linewidths=0.5,
    linecolor="gray"
)
axes[0].set_title("Matriz de Confusión - Conjunto de Prueba", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Predicción", fontsize=11)
axes[0].set_ylabel("Real", fontsize=11)

# --- Estadísticas por clase ---
report = classification_report(true_classes, pred_classes, target_names=class_names, output_dict=True)
report_df = pd.DataFrame(report).T.drop("accuracy", errors="ignore")

report_df[["precision", "recall", "f1-score", "support"]] = report_df[
    ["precision", "recall", "f1-score", "support"]
].apply(pd.to_numeric, errors="coerce")

sns.heatmap(
    report_df[["precision", "recall", "f1-score"]].iloc[:-2],
    annot=True,
    fmt=".3f",
    cmap="YlGnBu",
    ax=axes[1],
    linewidths=0.5,
    linecolor="gray",
    vmin=0,
    vmax=1
)
axes[1].set_title("Métricas por Clase", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Métrica", fontsize=11)
axes[1].set_ylabel("Clase", fontsize=11)

plt.tight_layout()
plt.show()

# Imprimir reporte completo
print(f"\nAccuracy global en prueba: {test_accuracy:.4f}")
print(f"Baseline (clase mayoritaria): {majority_baseline:.4f}\n")
print("Reporte de clasificación completo:")
print(classification_report(true_classes, pred_classes, target_names=class_names, digits=4))

In [ ]:
# Guardar el modelo entrenado
model.save("ann_multiclass_model.keras")
print("Modelo guardado como 'ann_multiclass_model.keras'")


Modelo guardado como 'ann_multiclass_model.keras'
